# 02 — Exploratory Data Analysis dengan Spark

Notebook ini membaca data dari MinIO (raw zone) dan melakukan eksplorasi awal menggunakan PySpark sebelum masuk ke tahap data mining.

## 1. Setup SparkSession

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
spark = create_spark_session("02 - EDA")
print(f"Spark version: {spark.version}")

## 2. Baca Data dari MinIO

In [ ]:
BUCKET = "datalake"

# Gunakan wildcard untuk membaca semua partisi tanggal
df_customers = spark.read.csv(f"s3a://{BUCKET}/raw/rdbms/customers/*/customers_from_db.csv", header=True, inferSchema=True)
df_orders    = spark.read.csv(f"s3a://{BUCKET}/raw/rdbms/orders/*/orders_from_db.csv",       header=True, inferSchema=True)
df_products  = spark.read.csv(f"s3a://{BUCKET}/raw/xlsx/products/*/products_from_xlsx.csv",  header=True, inferSchema=True)

print(f"customers : {df_customers.count()} baris")
print(f"orders    : {df_orders.count()} baris")
print(f"products  : {df_products.count()} baris")

## 3. Schema & Sample Data

In [ ]:
df_customers.printSchema()
df_customers.show(5)

In [ ]:
df_orders.printSchema()
df_orders.show(5)

In [ ]:
df_products.printSchema()
df_products.show(5)

## 4. Statistik Deskriptif

In [ ]:
print("=== Statistik Customers ===")
df_customers.describe("age").show()

print("=== Statistik Orders ===")
df_orders.describe("quantity").show()

print("=== Statistik Products ===")
df_products.describe("price", "stock").show()

## 5. Distribusi per Kolom Kategorik

In [ ]:
print("=== Distribusi Kota ===")
df_customers.groupBy("city").count().orderBy("count", ascending=False).show()

print("=== Distribusi Kategori Produk ===")
df_products.groupBy("category").count().orderBy("count", ascending=False).show()

## 6. Feature Engineering — Fitur Pelanggan

In [ ]:
from analysis.preprocessing import build_customer_features

df_features = build_customer_features(df_customers, df_orders, df_products)
df_features.show(10)
df_features.describe("age", "total_orders", "total_spend").show()

## 7. Visualisasi (Pandas)

Konversi ke Pandas untuk visualisasi karena dataset kecil.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pdf = df_features.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pdf["age"].hist(bins=10, ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi Usia")

pdf["total_orders"].hist(bins=10, ax=axes[1], color="seagreen")
axes[1].set_title("Distribusi Total Orders")

pdf["total_spend"].hist(bins=10, ax=axes[2], color="darkorange")
axes[2].set_title("Distribusi Total Spend")

plt.tight_layout()
plt.show()

## 8. Simpan Fitur ke MinIO (processed zone)

In [ ]:
df_features.write.mode("overwrite").option("header", True) \n    .csv(f"s3a://{BUCKET}/processed/customer_features/")
print("Fitur pelanggan berhasil disimpan ke processed zone.")